# 📗 Bài 4 — Tự xây Mini-GPT

**Tuần 15** · 🏆 **Đỉnh cao của Phase 06**

Bạn sắp ghép mọi thứ đã học thành một GPT hoàn chỉnh và train nó sinh tiếng Việt.

**Trên CPU, toàn bộ mất khoảng 1 phút.**

In [ ]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

MAU = ["#4269d0", "#efb118", "#3ca951", "#ff725c", "#a463f2"]
MAU_CHINH, MAU_NHAN = MAU[0], MAU[3]
torch.manual_seed(1337)

DATA = Path("../../../data")
if not (DATA / "van_ban_viet.txt").exists():
    DATA = Path("data")

van_ban = (DATA / "van_ban_viet.txt").read_text(encoding="utf-8")
print(f"Kho ngữ liệu: {len(van_ban):,} ký tự")
print(van_ban[:150])

## 1. Tokenizer & dữ liệu

In [ ]:
chars = sorted(set(van_ban))
V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}

ma_hoa = lambda s: [stoi[c] for c in s]
giai_ma = lambda ids: "".join(itos[int(i)] for i in ids)

du_lieu = torch.tensor(ma_hoa(van_ban), dtype=torch.long)
n = int(0.9 * len(du_lieu))
train_data, val_data = du_lieu[:n], du_lieu[n:]

print(f"  Từ vựng    : {V} ký tự")
print(f"  Train / Val: {len(train_data):,} / {len(val_data):,} token")

## 2. Batch — input và target lệch nhau MỘT token

In [ ]:
BLOCK = 64        # context window
BATCH = 32
D_MODEL = 96
N_HEAD = 4
N_LAYER = 3


def lay_batch(split, seed=None):
    d = train_data if split == "train" else val_data
    g = torch.Generator().manual_seed(seed) if seed is not None else None
    ix = torch.randint(len(d) - BLOCK - 1, (BATCH,), generator=g)
    x = torch.stack([d[i : i + BLOCK] for i in ix])
    y = torch.stack([d[i + 1 : i + BLOCK + 1] for i in ix])
    return x, y


xb, yb = lay_batch("train", seed=0)
print(f"  x {tuple(xb.shape)}   y {tuple(yb.shape)}")
print()
print(f"  x[0][:20]: {giai_ma(xb[0][:20])!r}")
print(f"  y[0][:20]: {giai_ma(yb[0][:20])!r}")
print()
print("  -> y chỉ là x DỊCH SANG TRÁI một token.")
print("     Nhiệm vụ của model: cho x, đoán y. Tức là 'đoán ký tự tiếp theo'.")

In [ ]:
# Nhờ causal mask, MỘT chuỗi cho ta BLOCK bài toán dự đoán cùng lúc
print("  Từ một chuỗi, model học được nhiều ví dụ:")
for t in range(1, 6):
    ngu_canh = giai_ma(xb[0][:t])
    dich = itos[int(yb[0][t - 1])]
    print(f"    thấy {ngu_canh!r:<22} -> phải đoán {dich!r}")

## 3. ⭐ Xây từng khối

In [ ]:
class Head(nn.Module):
    """Một đầu self-attention có causal mask."""

    def __init__(self, d_model, d_k):
        super().__init__()
        self.q = nn.Linear(d_model, d_k, bias=False)
        self.k = nn.Linear(d_model, d_k, bias=False)
        self.v = nn.Linear(d_model, d_k, bias=False)
        self.d_k = d_k
        self.register_buffer("mask", torch.tril(torch.ones(BLOCK, BLOCK)))

    def forward(self, x):
        B, T, _ = x.shape
        q, k, v = self.q(x), self.k(x), self.v(x)

        diem = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)
        diem = diem.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        w = torch.softmax(diem, dim=-1)

        return w @ v, w


h = Head(D_MODEL, 24)
ra, w = h(torch.randn(2, 10, D_MODEL))
print(f"  Head: vào (2,10,{D_MODEL}) -> ra {tuple(ra.shape)}, trọng số {tuple(w.shape)}")

> `register_buffer` đăng ký mask là một phần của model (được lưu/nạp cùng, chuyển GPU cùng) nhưng **không phải tham số học được**. Đúng bản chất: mask là hằng số.

In [ ]:
class MultiHead(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.heads = nn.ModuleList([Head(d_model, d_model // n_head) for _ in range(n_head)])
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        ket_qua = [h(x) for h in self.heads]
        noi = torch.cat([r for r, _ in ket_qua], dim=-1)
        return self.proj(noi), torch.stack([w for _, w in ket_qua], dim=1)


class FeedForward(nn.Module):
    """Mỗi token tự 'suy nghĩ' thêm. Rộng gấp 4 lần — quy ước từ bài báo gốc."""

    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.attn = MultiHead(d_model, n_head)
        self.ff = FeedForward(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        ra, w = self.attn(self.ln1(x))
        x = x + ra                      # skip connection
        x = x + self.ff(self.ln2(x))    # skip connection
        return x, w


b = Block(D_MODEL, N_HEAD)
ra, w = b(torch.randn(2, 10, D_MODEL))
print(f"  Block: ra {tuple(ra.shape)}, trọng số {tuple(w.shape)}  (B, đầu, T, T)")

```
     x ─────────────────────┐
     │                      │
  LayerNorm                 │
     │                      │
  Multi-head attention      │  ← token TRAO ĐỔI thông tin
     │                      │
     +──────────────────────┘  skip connection
     │
     ├─────────────────────┐
     │                     │
  LayerNorm                │
     │                     │
  Feed-forward             │  ← mỗi token TỰ suy nghĩ
     │                     │
     +─────────────────────┘  skip connection
```

| Thành phần | Giải vấn đề gì |
|---|---|
| **Attention** | Token trao đổi thông tin với nhau |
| **Feed-forward** | Mỗi token tự "suy nghĩ" thêm (rộng gấp 4) |
| **Skip connection** | Gradient chảy được qua mạng sâu |
| **LayerNorm** | Ổn định giá trị, train không phân kỳ |

> 💡 **Skip connection** (`x + f(x)`) là phát minh từ ResNet bạn đã gặp ở Phase 05. Không có nó, mạng 50 lớp không train nổi.

## 4. ⭐⭐ GPT hoàn chỉnh

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab, d_model, n_head, n_layer, block):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, d_model)
        self.pos_emb = nn.Embedding(block, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab)

    def forward(self, idx, tra_ve_attention=False):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))

        cac_w = []
        for blk in self.blocks:
            x, w = blk(x)
            cac_w.append(w)

        logits = self.head(self.ln_f(x))
        return (logits, cac_w) if tra_ve_attention else logits

    @torch.no_grad()
    def sinh(self, idx, so_token_moi, temperature=0.8, top_k=None):
        self.eval()
        for _ in range(so_token_moi):
            logits = self(idx[:, -BLOCK:])[:, -1, :] / temperature
            if top_k is not None:
                nguong = torch.topk(logits, top_k).values[:, -1:]
                logits = logits.masked_fill(logits < nguong, float("-inf"))
            xac_suat = torch.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(xac_suat, 1)], dim=1)
        return idx


torch.manual_seed(1337)
model = MiniGPT(V, D_MODEL, N_HEAD, N_LAYER, BLOCK)
tong = sum(p.numel() for p in model.parameters())
print(f"  Tổng tham số: {tong:,}")

In [ ]:
# Tham số nằm ở đâu?
nhom = {}
for ten, p in model.named_parameters():
    khoa = ten.split(".")[0]
    nhom[khoa] = nhom.get(khoa, 0) + p.numel()

print(f"  {'thành phần':<12} {'tham số':>10} {'%':>7}")
print("  " + "-" * 32)
for k, v in sorted(nhom.items(), key=lambda x: -x[1]):
    print(f"  {k:<12} {v:>10,} {v/tong:>6.1%}")

## 5. Sinh văn bản TRƯỚC khi train

In [ ]:
bat_dau = torch.tensor([[stoi["N"]]], dtype=torch.long)
truoc = giai_ma(model.sinh(bat_dau, 200, temperature=0.8)[0])

print("--- MODEL CHƯA TRAIN ---")
print(truoc)
print()
print("  Rác hoàn toàn. Trọng số còn ngẫu nhiên.")

## 6. ⭐ Train

In [ ]:
@torch.no_grad()
def uoc_luong_loss(model, so_lan=20):
    model.eval()
    kq = {}
    for split in ("train", "val"):
        ls = []
        for i in range(so_lan):
            x, y = lay_batch(split, seed=1000 + i)
            logits = model(x)
            ls.append(F.cross_entropy(logits.view(-1, V), y.view(-1)).item())
        kq[split] = float(np.mean(ls))
    model.train()
    return kq


SO_BUOC = 1500
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)

lich_su = []
t0 = time.perf_counter()

for buoc in range(SO_BUOC + 1):
    xb, yb = lay_batch("train")

    logits = model(xb)                                          # ① forward
    loss = F.cross_entropy(logits.view(-1, V), yb.view(-1))     # ② loss
    opt.zero_grad()                                             # ③ xoá grad
    loss.backward()                                             #    backward
    opt.step()                                                  # ④ update

    if buoc % 150 == 0:
        l = uoc_luong_loss(model)
        lich_su.append((buoc, l["train"], l["val"]))
        print(f"  bước {buoc:>4}   train {l['train']:.4f}   val {l['val']:.4f}   "
              f"({time.perf_counter()-t0:.0f}s)")

print(f"\n  Xong trong {time.perf_counter()-t0:.0f} giây trên CPU.")

In [ ]:
b, tr, va = zip(*lich_su)
plt.plot(b, tr, lw=2, color=MAU[0], label="train")
plt.plot(b, va, lw=2, color=MAU[3], label="validation")
plt.axhline(math.log(V), color="gray", ls="--",
            label=f"đoán mò = ln({V}) = {math.log(V):.2f}")
plt.xlabel("bước"); plt.ylabel("cross-entropy loss")
plt.title("Đường cong học của mini-GPT")
plt.legend(frameon=False)
plt.show()

print(f"  Loss ban đầu ~{math.log(V):.2f} (đoán mò đều {V} ký tự)")
print(f"  Loss cuối    ~{va[-1]:.2f}")
print(f"  Perplexity   = e^{va[-1]:.2f} = {math.exp(va[-1]):.1f}")
print(f"  -> Model 'phân vân' giữa ~{math.exp(va[-1]):.0f} ký tự thay vì {V}.")

**Perplexity** = `e^loss` — đọc là *"model đang phân vân giữa bao nhiêu lựa chọn"*.

Loss ban đầu bằng đúng `ln(vocab_size)` vì model đoán đều tất cả ký tự. Đây là **baseline** của bài toán mô hình ngôn ngữ — cùng tinh thần với `DummyClassifier` ở Phase 04.

> 📌 Nếu loss của bạn không giảm dưới `ln(V)`, model chưa học được gì cả.

## 7. 🏆 Sinh văn bản SAU khi train

In [ ]:
sau = giai_ma(model.sinh(bat_dau, 500, temperature=0.8)[0])
print("--- MODEL ĐÃ TRAIN ---")
print(sau)

> 🏆 **Bạn vừa tự xây một GPT và nó sinh được tiếng Việt đúng chính tả, đúng dấu, đúng dấu câu.**
>
> Không ai dạy nó luật chính tả tiếng Việt. Nó học từ 264 KB văn bản, chỉ bằng cách đoán ký tự tiếp theo, lặp lại 1.500 lần.

**Một lưu ý thành thật:** kho ngữ liệu này được sinh từ mẫu câu nên **lặp lại nhiều hơn** văn bản thật. Mini-GPT học nó rất nhanh và sinh ra câu khá "sạch" — đẹp hơn so với khi train trên dữ liệu thật cùng kích thước. Đó là đánh đổi để bài học chạy được offline trong một phút.

## 8. Temperature — cái nút bạn sẽ gặp lại ở Phase 07

In [ ]:
for temp in [0.2, 0.8, 1.5]:
    torch.manual_seed(0)
    vb = giai_ma(model.sinh(bat_dau, 180, temperature=temp)[0])
    print(f"--- temperature = {temp} ---")
    print(vb.replace("\n", " ")[:180])
    print()

In [ ]:
# Nhìn temperature làm gì với phân phối xác suất
ngu_canh = torch.tensor([ma_hoa("Nam thích ")], dtype=torch.long)
with torch.no_grad():
    logits = model(ngu_canh)[0, -1, :]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
for ax, temp in zip(axes, [0.2, 0.8, 1.5]):
    p = torch.softmax(logits / temp, dim=-1)
    top = torch.topk(p, 10)
    ax.bar(range(10), top.values, color=MAU_CHINH)
    ax.set_xticks(range(10), [repr(itos[int(i)])[1:-1] for i in top.indices], fontsize=8)
    ax.set_title(f"temperature = {temp}", fontsize=10)
    ax.set_ylim(0, 1)
    if temp == 0.2:
        ax.set_ylabel("xác suất")

plt.suptitle('Ký tự tiếp theo sau "Nam thích "', y=1.04)
plt.tight_layout(); plt.show()

| Temperature | Hiệu ứng |
|---|---|
| `→ 0` | Luôn chọn token cao nhất — ổn định nhưng **lặp lại** |
| `0.7–0.9` | Cân bằng — mặc định thông dụng |
| `> 1.2` | Rất ngẫu nhiên — sáng tạo nhưng dễ **vô nghĩa** |

**Cơ chế:** chia logits cho temperature **trước** softmax. Chia cho số nhỏ → khoảng cách giữa các logit bị phóng to → phân phối nhọn hơn.

> 🔗 **Đây chính xác là tham số `temperature` trong Claude API** mà bạn sẽ dùng ở Phase 07. Giờ bạn biết nó làm gì về mặt cơ học — không phải "độ sáng tạo" mơ hồ, mà là một phép chia trước softmax.

**Các chiến lược lấy mẫu khác:**

| Cách | Ý tưởng |
|---|---|
| **Greedy** | Luôn lấy token cao nhất — dễ lặp vô hạn |
| **Top-k** | Chỉ lấy mẫu trong `k` token cao nhất |
| **Top-p (nucleus)** | Lấy mẫu trong nhóm nhỏ nhất có tổng xác suất ≥ `p` |

## 9. ⭐ Nhìn vào attention của model ĐÃ TRAIN

In [ ]:
doan = "Nam thích đọc sách vào buổi tối."
ids = torch.tensor([ma_hoa(doan)], dtype=torch.long)

with torch.no_grad():
    model.eval()
    _, cac_w = model(ids, tra_ve_attention=True)

nhan = list(doan)
lop_cuoi = cac_w[-1][0]        # (n_head, T, T)

fig, axes = plt.subplots(1, N_HEAD, figsize=(16, 4))
for i, ax in enumerate(axes):
    ax.imshow(lop_cuoi[i], cmap="Blues", vmin=0)
    ax.set_xticks(range(len(nhan)), nhan, fontsize=7)
    ax.set_yticks(range(len(nhan)), nhan, fontsize=7)
    ax.set_title(f"lớp {N_LAYER} — đầu {i+1}", fontsize=10)
    ax.grid(visible=False)

plt.suptitle("Attention của model ĐÃ TRAIN — mỗi đầu học một kiểu chú ý", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# So sánh attention giữa các LỚP
fig, axes = plt.subplots(1, N_LAYER, figsize=(14, 4))
for lop, ax in enumerate(axes):
    tb = cac_w[lop][0].mean(dim=0)          # trung bình các đầu
    ax.imshow(tb, cmap="Blues", vmin=0)
    ax.set_xticks(range(len(nhan)), nhan, fontsize=6)
    ax.set_yticks(range(len(nhan)), nhan, fontsize=6)
    ax.set_title(f"lớp {lop+1} (trung bình các đầu)", fontsize=10)
    ax.grid(visible=False)
plt.tight_layout(); plt.show()

print("  Tam giác trên phải LUÔN trắng — causal mask đang hoạt động.")
print("  Lớp đầu thường chú ý cục bộ (ký tự liền kề);")
print("  lớp sau chú ý xa hơn, bắt cấu trúc dài hơn.")

## 10. Mini-GPT của bạn vs GPT-4

In [ ]:
print(f"  {'':<22} {'mini-GPT của bạn':>20} {'GPT-4 (ước tính)':>22}")
print("  " + "-" * 66)
for ten, a, b in [
    ("Tham số", f"{tong:,}", "~1.000.000.000.000"),
    ("Số lớp", str(N_LAYER), "~100+"),
    ("Số đầu attention", str(N_HEAD), "~128"),
    ("Context window", f"{BLOCK} token", "128.000+ token"),
    ("Từ vựng", f"{V} ký tự", "~100.000 subword"),
    ("Dữ liệu train", f"{len(van_ban)/1000:.0f} KB", "~10 TB"),
    ("Chi phí train", "64 giây CPU", "hàng chục triệu đô"),
]:
    print(f"  {ten:<22} {a:>20} {b:>22}")

print()
print("  KIẾN TRÚC thì giống nhau về bản chất:")
print("    embedding -> N x [attention + feed-forward] -> logits")

## 11. Tổng kết Phase 06

```
tokens
  │
Token embedding + Position embedding
  │
Transformer Block × N     ← attention + feed-forward + skip + layernorm
  │
LayerNorm
  │
Linear (d → vocab)
  │
logits → softmax → xác suất token tiếp theo
```

**Đó là toàn bộ GPT.** Bạn vừa xây nó.

### 🔗 Cầu nối sang Phase 07

Mọi khái niệm bạn sẽ gặp ở Phase 07 giờ đã có ý nghĩa cơ học:

| Khái niệm LLM API | Bạn đã thấy nó ở đâu |
|---|---|
| **Token** & chi phí | Bài 1 — tokenizer, và tiếng Việt tốn token hơn |
| **Context window** | `BLOCK = 64` — giới hạn của `pos_emb` |
| **Temperature** | Bài 4 — phép chia trước softmax |
| **Streaming** | Vòng lặp `sinh()` — model sinh **từng token một** |
| **Embedding / RAG** | Bài 2 — cosine similarity |
| **Vì sao LLM "ảo giác"** | Nó chỉ đoán token tiếp theo, không tra cứu sự thật |
| **Vì sao LLM sai khi đếm chữ cái** | Nó thấy **token**, không thấy ký tự |

> 🎯 **Điểm mấu chốt:** LLM không "hiểu" hay "biết" gì cả. Nó dự đoán token tiếp theo dựa trên thống kê — chính xác như mini-GPT bạn vừa train. Mọi kỹ thuật ở Phase 07–09 (prompt, RAG, agent) đều là cách **định hình đầu vào** để phép dự đoán đó cho ra thứ ta muốn.

## 🎯 Project P4

Xong bài này → làm [**P4 — Mini-GPT**](../../../projects/P4-mini-gpt/README.md).